# Preprocesamiento para el modelo

Ingeniería de características sobre las features seleccionadas en `1.0-jec-analisis-exploratorio`: imputación, codificación, tratamiento de outliers, transformación y escalado. Cada paso se aplica primero de forma individual para revisar su efecto y después se integra en un `Pipeline` de sklearn construido con transformadores de feature-engine. El resultado son los datasets procesados que alimentan el notebook de modelos.

## 1. Librerías

Importación de las librerías de manipulación de datos, de los transformadores de feature-engine y de los componentes de sklearn.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from feature_engine.encoding import OrdinalEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Configuración y carga de datos

Lectura de `pipeline_config.yaml` para obtener la variable objetivo y las features por tipo, definición de rutas y parámetros del preprocesamiento, y carga de los datos crudos.

In [2]:
# Rutas configurables: cambia estas variables si las fuentes o destinos se mueven
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_FILE = PROJECT_ROOT / "pipeline_config.yaml"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

# Parámetros del preprocesamiento
TEST_SIZE = 0.2
RANDOM_STATE = 42
IQR_FOLD = 1.5
SKEW_THRESHOLD = 0.75

with open(CONFIG_FILE, encoding="utf-8") as f:
    config = yaml.safe_load(f)

TARGET = config["TARGET"]
MODEL_FEATURES = config["MODEL_FEATURES"]
CONTINUOUS_FEATURES = config["CONTINUOUS_FEATURES"]
DISCRETE_FEATURES = config["DISCRETE_FEATURES"]
CATEGORICAL_FEATURES = config["CATEGORICAL_FEATURES"]
NUMERIC_FEATURES = CONTINUOUS_FEATURES + DISCRETE_FEATURES

train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

print(f"train: {train.shape}, test: {test.shape}, features: {len(MODEL_FEATURES)}")

train: (1460, 81), test: (1459, 80), features: 30


## 3. División en entrenamiento y validación

Separación de `train.csv` en entrenamiento (80 %) y validación (20 %). Todos los transformadores se ajustan únicamente con el conjunto de entrenamiento para evitar fuga de información. La variable objetivo se transforma con `log1p` para corregir su sesgo a la derecha; las predicciones se revierten con `expm1`.

In [4]:
X = train[MODEL_FEATURES]
y = np.log1p(train[TARGET])
X_test = test[MODEL_FEATURES]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}, X_test: {X_test.shape}")
print(f"Sesgo del target: original {train[TARGET].skew():.2f}, log1p {y.skew():.2f}")

X_train: (1168, 30), X_valid: (292, 30), X_test: (1459, 30)
Sesgo del target: original 1.88, log1p 0.12


## 4. Imputación de variables categóricas

Reemplazo de los valores faltantes de las variables categóricas por la categoría más frecuente (moda) de cada variable. Se aplica a todas las categóricas, ya que `test.csv` presenta faltantes en variables que están completas en entrenamiento.

In [5]:
print("Faltantes antes:", X_train[CATEGORICAL_FEATURES].isna().sum()[lambda s: s > 0].to_dict())

cat_imputer = CategoricalImputer(imputation_method="frequent", variables=CATEGORICAL_FEATURES)
X_train_t = cat_imputer.fit_transform(X_train)

print("Moda utilizada:", cat_imputer.imputer_dict_)
print("Faltantes después:", X_train_t[CATEGORICAL_FEATURES].isna().sum().sum())

Faltantes antes: {'BsmtQual': 28, 'GarageFinish': 64, 'FireplaceQu': 547, 'GarageType': 64, 'BsmtFinType1': 28}
Moda utilizada: {'Neighborhood': 'NAmes', 'ExterQual': 'TA', 'BsmtQual': 'TA', 'KitchenQual': 'TA', 'GarageFinish': 'Unf', 'FireplaceQu': 'Gd', 'Foundation': 'PConc', 'GarageType': 'Attchd', 'BsmtFinType1': 'Unf', 'HeatingQC': 'Ex'}
Faltantes después: 0


## 5. Imputación de variables numéricas

Reemplazo de los valores faltantes de las variables continuas y discretas por la mediana, estadístico robusto ante el sesgo y los valores extremos. Este paso es necesario porque el tratamiento de outliers, la transformación y el escalado no admiten valores nulos.

In [6]:
print("Faltantes antes:", X_train_t[NUMERIC_FEATURES].isna().sum()[lambda s: s > 0].to_dict())

num_imputer = MeanMedianImputer(imputation_method="median", variables=NUMERIC_FEATURES)
X_train_t = num_imputer.fit_transform(X_train_t)

print("Faltantes después:", X_train_t[NUMERIC_FEATURES].isna().sum().sum())

Faltantes antes: {'GarageYrBlt': 64, 'MasVnrArea': 6, 'LotFrontage': 217}
Faltantes después: 0


## 6. Codificación de variables categóricas

Codificación ordinal ordenada por el target: las categorías de cada variable se ordenan según la media de `SalePrice` (en escala logarítmica) y se les asigna un entero de 0 a k-1. Genera una sola columna por variable y conserva la relación monótona con el precio. Las categorías no vistas en entrenamiento se codifican como -1.

In [7]:
encoder = OrdinalEncoder(encoding_method="ordered", variables=CATEGORICAL_FEATURES, unseen="encode")
X_train_t = encoder.fit_transform(X_train_t, y_train)

print("Ejemplo de codificación (ExterQual):", encoder.encoder_dict_["ExterQual"])
X_train_t[CATEGORICAL_FEATURES].head()

Ejemplo de codificación (ExterQual): {'Fa': 0, 'TA': 1, 'Gd': 2, 'Ex': 3}


,Neighborhood,ExterQual,BsmtQual,KitchenQual,GarageFinish,FireplaceQu,Foundation,GarageType,BsmtFinType1,HeatingQC
254,9,1,1,1,1,1,2,4,0,2
1066,15,2,2,1,1,3,5,4,4,3
638,4,1,0,1,0,1,2,4,4,3
799,8,1,2,2,0,3,1,1,3,4
380,8,1,1,2,0,1,1,1,1,2


## 7. Tratamiento de outliers

Capping de las variables continuas con el método del rango intercuartílico: los valores fuera de [Q1 − 1.5·IQR, Q3 + 1.5·IQR] se reemplazan por el límite correspondiente. No se eliminan registros, por lo que el paso se aplica igual en entrenamiento, validación y predicción.

In [8]:
winsorizer = Winsorizer(capping_method="iqr", tail="both", fold=IQR_FOLD, variables=CONTINUOUS_FEATURES)
X_train_capped = winsorizer.fit_transform(X_train_t)

capped_counts = (X_train_capped[CONTINUOUS_FEATURES] != X_train_t[CONTINUOUS_FEATURES]).sum()
limits = pd.DataFrame({
    "limite_inferior": winsorizer.left_tail_caps_,
    "limite_superior": winsorizer.right_tail_caps_,
    "valores_recortados": capped_counts,
})
X_train_t = X_train_capped
limits

,limite_inferior,limite_superior,valores_recortados
GrLivArea,176.375,2761.375,23
GarageArea,-11.500,928.500,18
TotalBsmtSF,41.125,2054.125,47
1stFlrSF,124.875,2159.875,13
YearBuilt,1881.000,2073.000,5
YearRemodAdd,1909.000,2061.000,0
GarageYrBlt,1903.500,2059.500,1
MasVnrArea,-249.000,415.000,77
BsmtFinSF1,-1081.500,1802.500,5
LotFrontage,31.500,107.500,85


## 8. Transformación de variables continuas

Aplicación de la transformación Yeo-Johnson a las variables continuas cuyo sesgo absoluto, medido después del capping, supera `SKEW_THRESHOLD`. Yeo-Johnson admite ceros y valores negativos y estima la transformación óptima de cada variable para aproximarla a una distribución normal.

In [9]:
skewness = X_train_t[CONTINUOUS_FEATURES].skew()
SKEWED_FEATURES = skewness[skewness.abs() > SKEW_THRESHOLD].index.tolist()
print(f"Variables con |skew| > {SKEW_THRESHOLD}: {SKEWED_FEATURES}")

yeo_johnson = YeoJohnsonTransformer(variables=SKEWED_FEATURES)
X_train_t = yeo_johnson.fit_transform(X_train_t)

pd.DataFrame({
    "skew_antes": skewness[SKEWED_FEATURES],
    "skew_despues": X_train_t[SKEWED_FEATURES].skew(),
    "lambda": pd.Series(yeo_johnson.lambda_dict_),
})

Variables con |skew| > 0.75: ['MasVnrArea']


,skew_antes,skew_despues,lambda
MasVnrArea,1.274889,0.374998,-0.218259


## 9. Escalado de variables

Estandarización de todas las features con `StandardScaler` (media 0 y desviación estándar 1), integrado en feature-engine mediante `SklearnTransformerWrapper` para conservar el formato DataFrame.

In [10]:
scaler = SklearnTransformerWrapper(transformer=StandardScaler(), variables=MODEL_FEATURES)
X_train_t = scaler.fit_transform(X_train_t)

X_train_t.describe().loc[["mean", "std"]].round(2)

,GrLivArea,GarageArea,TotalBsmtSF,1stFlrSF,YearBuilt,YearRemodAdd,GarageYrBlt,MasVnrArea,BsmtFinSF1,LotFrontage,...,Neighborhood,ExterQual,BsmtQual,KitchenQual,GarageFinish,FireplaceQu,Foundation,GarageType,BsmtFinType1,HeatingQC
mean,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,-0.0,0.0,...,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 10. Pipeline de preprocesamiento

Integración de todos los pasos anteriores en un `Pipeline` de sklearn, ajustado con el conjunto de entrenamiento. Se verifica que su salida coincide con la aplicación paso a paso.

In [11]:
preprocessing_pipeline = Pipeline([
    ("imputacion_categoricas", CategoricalImputer(imputation_method="frequent", variables=CATEGORICAL_FEATURES)),
    ("imputacion_numericas", MeanMedianImputer(imputation_method="median", variables=NUMERIC_FEATURES)),
    ("codificacion", OrdinalEncoder(encoding_method="ordered", variables=CATEGORICAL_FEATURES, unseen="encode")),
    ("outliers", Winsorizer(capping_method="iqr", tail="both", fold=IQR_FOLD, variables=CONTINUOUS_FEATURES)),
    ("transformacion", YeoJohnsonTransformer(variables=SKEWED_FEATURES)),
    ("escalado", SklearnTransformerWrapper(transformer=StandardScaler(), variables=MODEL_FEATURES)),
])

preprocessing_pipeline.fit(X_train, y_train)

X_train_proc = preprocessing_pipeline.transform(X_train)
X_valid_proc = preprocessing_pipeline.transform(X_valid)
X_test_proc = preprocessing_pipeline.transform(X_test)

print("Coincide con el paso a paso:", np.allclose(X_train_proc.values, X_train_t.values))
print("Nulos (train, valid, test):", [d.isna().sum().sum() for d in (X_train_proc, X_valid_proc, X_test_proc)])
preprocessing_pipeline

Coincide con el paso a paso: True
Nulos (train, valid, test): [np.int64(0), np.int64(0), np.int64(0)]


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputacion_categoricas', ...), ('imputacion_numericas', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,imputation_method,'frequent'
,variables,"['Neighborhood', 'ExterQual', ...]"
,fill_value,'Missing'
,return_object,False


## 11. Generación de datasets procesados

Exportación de los datasets transformados a `data/processed/`, que serán el input del notebook de modelos:

- `train_processed.csv` y `valid_processed.csv`: features procesadas y `SalePrice` en escala `log1p`.
- `test_processed.csv`: `Id` y features procesadas, sin target.

El pipeline ajustado se guarda en `models/preprocessing_pipeline.joblib` para transformar datos nuevos con los mismos parámetros.

In [12]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train_proc.assign(**{TARGET: y_train}).to_csv(PROCESSED_DIR / "train_processed.csv", index=False)
X_valid_proc.assign(**{TARGET: y_valid}).to_csv(PROCESSED_DIR / "valid_processed.csv", index=False)
X_test_proc.insert(0, "Id", test["Id"])
X_test_proc.to_csv(PROCESSED_DIR / "test_processed.csv", index=False)

joblib.dump(preprocessing_pipeline, MODELS_DIR / "preprocessing_pipeline.joblib")

for file in sorted(PROCESSED_DIR.glob("*.csv")) + [MODELS_DIR / "preprocessing_pipeline.joblib"]:
    print(file.relative_to(PROJECT_ROOT))

data\processed\test_processed.csv
data\processed\train_processed.csv
data\processed\valid_processed.csv
models\preprocessing_pipeline.joblib
